## OCR Batch 돌리는 코드
- STAGE에 송장파일이 추가되면 STEAM을 통해 감지되고 감지된 파일들을 OCR하여 결과를 가지고 CORTEX를 통해 정답을 참조하여 출력된 결과 테이블에 저장하는 코드입니다.
- `Dacument parser` 사용하지 않았습니다.




### 라이브러리 및 설정

In [ ]:
import json
from snowflake.snowpark.context import get_active_session
from snowflake.core import Root
import os
from dotenv import load_dotenv
load_dotenv()

STAGE_NAME    = os.getenv("STAGE_NAME")
DB_SCHEMA     = os.getenv("DB_SCHEMA")
SEARCH_DB     = os.getenv("SEARCH_DB")
SEARCH_SCHEMA = os.getenv("SEARCH_SCHEMA")
SEARCH_SVC    = os.getenv("SEARCH_SVC")

session = get_active_session()
print("세션 연결 완료")

### 프롬프트 설정

In [ ]:
def run_document_parse(stage_path: str, ocr_mode: str = "force"):
    options = json.dumps({
        "ocr": ocr_mode
    })

    sql = """
    CALL document_parse.app_public.document_parse(
        ?,
        ?
    )
    """

    result = session.sql(sql, params=[stage_path, options]).collect()[0][0]

    return json.loads(result)

def create_ocr_prompt(document_parser_result: str = "") -> str:
    return f"""
    You are an OCR assistant for invoice documents.
    ## Inputs:
    - An invoice image (ground truth).
    - Document Parser output (HTML): structural hint only.

    ## Critical:
    - The HTML is NOT ground truth. It may contain OCR mistakes.
    - Always verify and correct values using the IMAGE.
    - If HTML conflicts with the image, trust the image.

    --------------------------------
    Document Parser HTML:
    {document_parser_result}
    --------------------------------

    ## Task:
    - Extract ALL visible text from the IMAGE and output valid JSON.
    - Use the HTML only to help recover layout/structure (tables/sections).

    ## Structure rules:
    - Preserve the document's original layout and logical structure.
    - If a table has parent/child (hierarchical) headers, output nested JSON objects (no flattening).
    - Table data MUST be a list of row objects (even if one row). Do NOT create meta keys like "Headers", "Rows", etc.
    - Column labels must be used only as JSON keys; headers must not appear as data values.
    - If identical column names occur, make keys unique by adding a context-based suffix.

    ## Label rules:
    - Copy labels EXACTLY as shown (do not translate Korean).
    - Split slash-separated labels/values in order (A/B -> A, B).
    - Split multiple values in one cell using visible separators.

    ## Unclear text (VERY IMPORTANT):
    Append "(need_check)" to a VALUE if its cell/area contains anything other than clean printed text.
    Mark "(need_check)" when:
    - Any extra ink/mark/pattern/line exists within the SAME cell/area as the value
    - Any part of the characters is crossed, touched, or visually mixed with non-text pixels
    - The value is handwritten or manually filled
    - The text is faded, blurred, low-contrast, or partially ambiguous
    Rule of thumb: If the value is not visually isolated as "text-only" inside its cell, it MUST be "(need_check)".
    Do NOT apply to keys.

    ## Item category:
    - In "Header", include a fixed key "Item Category" classified as exactly one of:
      [폐기물, 토사, PHC파일, 고로슬래그, 철근, 레미콘, 알폼, PF보드, 경질우레탄보드, 석고보드, 드라이몰탈, 벌크시멘트]
      If unclear: "기타".

    ## Output:
    - Output ONLY valid JSON. No markdown. No explanations.
    """


def create_extraction_prompt(category_name: str, reference_context: str, input_text: str) -> str:
    return f"""
        당신은 데이터 구조화 전문 AI입니다.

        [목표]
        입력된 불완전한 텍스트를 분석하여, [참고 예시]와 완벽하게 동일한 구조(Key, Type)를 가진 최종 JSON을 생성하세요.

        [지침]
        1. 품목 카테고리는 '{category_name}' 입니다.
        2. 반드시 아래 [참고 예시]의 JSON 스키마를 따르세요. 없는 필드는 null로 두세요.
        3. [입력 데이터]의 오타나 비표준어는 [참고 예시]의 패턴을 보고 표준어로 보정하세요.

        [참고 예시 (Golden Standard)]
        {reference_context}

        [입력 데이터 (1차 OCR 결과)]
        {input_text}

        [출력 (Only JSON)]
    """

print("프롬프트 함수 정의 완료")

### 유틸 메서드

In [ ]:
def parse_json_safe(raw_text: str) -> dict:

    if not raw_text:
        return {"parse_error": "빈 응답", "raw_text": ""}
    
    cleaned = raw_text.replace("```json", "").replace("```", "").strip()
    
    try:
        obj = json.loads(cleaned)
        # 이중 직렬화 방어 (문자열 안에 JSON이 또 있는 경우)
        if isinstance(obj, str):
            obj = json.loads(obj.strip())
        return obj
    except json.JSONDecodeError as e:
        return {"parse_error": str(e), "raw_text": cleaned}


def extract_category(ocr_obj: dict) -> str:
    """OCR JSON 객체에서 Item Category 추출."""
    try:
        header = ocr_obj.get("Header") or ocr_obj.get("header") or {}
        category = (
            header.get("Item Category")
            or header.get("item_category")
            or ""
        ).strip()
        return category if category else "기타"
    except Exception:
        return "기타"


def process_ocr_result(raw_text: str) -> tuple[dict, str, str]:
    """
    OCR RAW_TEXT → (ocr_obj, input_text_str, category_name)
    input_text_str : 추출 프롬프트에 넣을 문자열
    """
    ocr_obj       = parse_json_safe(raw_text)
    input_text    = json.dumps(ocr_obj, ensure_ascii=False)
    category_name = extract_category(ocr_obj)
    return ocr_obj, input_text, category_name

print("유틸 함수 정의 완료")

### 배치 메서드 및 `Cortex Search` 호출

In [ ]:
def run_ocr_batch(session, model: str, ocr_prompt: str, stage_name: str) -> list:
    """
    Stream을 임시 테이블에 스냅샷으로 저장 후 OCR 수행.
    스트림은 이 시점에 1회만 소비되고, 이후 모든 처리는 스냅샷 기준으로 동작.
    """
    import uuid
    snapshot_table = f"{DB_SCHEMA}.SNAP_OCR_{uuid.uuid4().hex[:8].upper()}"

    # 스트림 → 임시 스냅샷 테이블 (스트림 소비 1회 확정)
    session.sql(f"""
        CREATE TEMPORARY TABLE {snapshot_table} AS
        SELECT RELATIVE_PATH
        FROM {STREAM_TABLE}
        WHERE METADATA$ACTION = 'INSERT'
    """).collect()

    # 스냅샷 기준으로 OCR 수행 (스트림 재참조 없음)
    sql = f"""
        SELECT
            RELATIVE_PATH,
            SNOWFLAKE.CORTEX.AI_COMPLETE(
                ?,
                ?,
                TO_FILE('{stage_name}/' || RELATIVE_PATH),
                OBJECT_CONSTRUCT('temperature', 0)
            ) AS RAW_TEXT
        FROM {snapshot_table}
    """
    return session.sql(sql, params=[model, ocr_prompt]).collect()


def run_extraction(session, model: str, extraction_prompt: str) -> dict | None:
    """
    CORTEX.COMPLETE로 최종 구조화 JSON 추출.
    실패 시 None 반환.
    """
    raw = session.sql(
        "SELECT SNOWFLAKE.CORTEX.COMPLETE(?, ?)",
        params=[model, extraction_prompt]
    ).collect()[0][0]
    
    result = parse_json_safe(raw)
    
    # parse_error 키가 있으면 실패로 간주
    if "parse_error" in result:
        return None
    return result


def get_reference_context(session, input_text: str, category_name: str) -> str:
    """Cortex Search에서 유사 카테고리 예시 JSON 검색."""
    root = Root(session)
    search_service = (
        root.databases[SEARCH_DB]
        .schemas[SEARCH_SCHEMA]
        .cortex_search_services[SEARCH_SVC]
    )
    
    NO_FILTER_CATEGORIES = {"기타", "Other", "미분류", ""}
    use_filter = category_name not in NO_FILTER_CATEGORIES
    
    search_params = {
        "query"  : input_text,
        "columns": ["final_json", "item_category"],
        "limit"  : 2,
    }
    if use_filter:
        search_params["filter"] = {"@eq": {"item_category": category_name}}
    
    try:
        resp = search_service.search(**search_params)
        
        # 필터 적용 시 결과 없으면 필터 없이 재시도
        if not resp.results and use_filter:
            del search_params["filter"]
            resp = search_service.search(**search_params)
        
        if not resp.results:
            return "참고할 과거 데이터가 없습니다. 일반적인 규칙에 따르세요."
        
        context = ""
        for i, row in enumerate(resp.results):
            found_cat = row.get("item_category", "알수없음")
            context += f"\n--- [참고 예시 {i+1} : (유사품목: {found_cat}) 정답] ---\n"
            context += json.dumps(row["final_json"], ensure_ascii=False) + "\n"
        return context
    
    except Exception as e:
        return f"검색 서비스 오류: {str(e)}"

print("Cortex 호출 함수 정의 완료")

### 결과반환 및 저장

In [ ]:
def process_single_row(session, model: str, stage_name: str, row) -> dict:
    """
    OCR 결과 row 1건을 받아 추출까지 수행 후 결과 dict 반환.
    예외는 내부에서 잡아 IS_SUCCESS=False로 기록 (배치 전체가 멈추지 않도록).
    """
    file_path  = row["RELATIVE_PATH"]
    path_parts = file_path.split("/")
    cd_site    = path_parts[0]  if path_parts else "UNKNOWN"
    image_name = path_parts[-1] if path_parts else "UNKNOWN"

    ocr_obj, input_text, category_name = process_ocr_result(row["RAW_TEXT"])
    raw_text_json_str = json.dumps(ocr_obj, ensure_ascii=False)

    final_result  = None
    error_message = None
    is_success    = False

    try:
        reference_context  = get_reference_context(session, input_text, category_name)
        extraction_prompt  = create_extraction_prompt(category_name, reference_context, input_text)
        final_result       = run_extraction(session, model, extraction_prompt)

        if final_result is None:
            raise ValueError("CORTEX.COMPLETE JSON 파싱 실패")

        is_success = True

    except Exception as e:
        error_message = str(e)

    # 처리 결과 로그 (노트북 확인용 → 프로시저에선 제거 가능)
    print(f"[{image_name}] 현장:{cd_site} | 카테고리:{category_name} | 성공:{is_success}")
    if not is_success:
        print(f"  └ 오류: {error_message}")

    return {
        "STAGE_LOCATION": stage_name,
        "CD_SITE"       : cd_site,
        "IMAGE_NAME"    : image_name,
        "ITEM_CATEGORY" : category_name,
        "RAW_TEXT"      : raw_text_json_str,
        "FINAL_JSON"    : json.dumps(final_result, ensure_ascii=False) if final_result else None,
        "IS_SUCCESS"    : is_success,
        "ERROR_MESSAGE" : error_message,
    }


def save_results(session, batch_results: list) -> int:
    if not batch_results:
        return 0
    import uuid
    temp_table = f"{DB_SCHEMA}.TEMP_OCR_{uuid.uuid4().hex[:8].upper()}"
    df = session.create_dataframe(batch_results)
    df.write.mode("overwrite").save_as_table(temp_table, table_type="temporary")

    inserted = session.sql(f"""
        SELECT COUNT(*) AS CNT FROM {temp_table} t
        WHERE NOT EXISTS (
            SELECT 1 FROM {RESULT_TABLE} r
            WHERE r.IMAGE_NAME = t.IMAGE_NAME
        )
    """).collect()[0]["CNT"]

    insert_sql = f"""
        INSERT INTO {RESULT_TABLE}
            (STAGE_LOCATION, CD_SITE, IMAGE_NAME, ITEM_CATEGORY,
             RAW_TEXT, FINAL_JSON, IS_SUCCESS, ERROR_MESSAGE, PROCESSED_AT)
        SELECT
            t.STAGE_LOCATION,
            t.CD_SITE,
            t.IMAGE_NAME,
            t.ITEM_CATEGORY,
            t.RAW_TEXT,
            TRY_PARSE_JSON(t.FINAL_JSON),
            t.IS_SUCCESS,
            t.ERROR_MESSAGE,
            CURRENT_TIMESTAMP()
        FROM {temp_table} t
        WHERE NOT EXISTS (
            SELECT 1 FROM {RESULT_TABLE} r
            WHERE r.IMAGE_NAME = t.IMAGE_NAME
        )
    """
    session.sql(insert_sql).collect()
    return inserted

print("처리/저장 함수 정의 완료")

### 메인 메서드

In [ ]:
def main(session, stage_name: str, model: str = MODEL) -> str:   
    ocr_prompt = create_ocr_prompt()
    ocr_results = run_ocr_batch(session, model, ocr_prompt, stage_name)
    if not ocr_results:
        return "처리할 신규 파일이 없습니다. (Stream Empty)"
    print(f"▶ OCR 완료: {len(ocr_results)}건 → 추출 시작\n")
    batch_results = [
        process_single_row(session, model, stage_name, row)
        for row in ocr_results
    ]
    success_cnt = sum(1 for r in batch_results if r["IS_SUCCESS"])
    fail_cnt    = len(batch_results) - success_cnt
    print(f"\n▶ 추출 완료 — 성공:{success_cnt} / 실패:{fail_cnt}")
    inserted = save_results(session, batch_results)
    return (
        f"총 {len(batch_results)}건 처리 "
        f"(성공:{success_cnt} / 실패:{fail_cnt}) | "
        f"실제 적재:{inserted}건"
    )

In [ ]:
# 신규로 테이블 추가되면 ocr하고, 완료되면 테이블에 저장되는 형태
# 신규로 추가되지 않으면 `처리할 신규 파일이 없습니다.`라는 메시지 출력
# ocr 실패하면 실패한 이미지 찾아서 ocr을 다시해야하는 구조라서 테이블에 전체 경로 컬럼까지 포함했습니다.

result = main(session, STAGE_NAME)
print("\n" + "="*50)
print(result)
print("="*50)